# 📊 UVR Index — Creazione Excel con Pesi da Inserire

---

Questo notebook legge il file Excel prodotto dalla pipeline Object Detection + Mapillary
e genera un nuovo Excel con:

- **Sezione A**: tabella pesi editabile (un peso UVR per ogni classe, da -1 a +1)
- **Sezione B**: conteggi per punto geografico (righe TOTALE) + FI_UVR calcolato con formula Excel

### Struttura attesa del file Excel di input
- Foglio: `Detection Results`
- Colonne fisse: `Latitudine`, `Longitudine`, `PANO_ID`, `Orientamento`
- Colonne classi (object detection + Mapillary): tutte le altre
- Riga TOTALE identificata dal valore `TOTALE` nella colonna `Orientamento`

### Come usare il file generato
1. Aprilo in Excel / Google Sheets
2. Nella **Sezione A** inserisci i pesi VBR per ogni classe (valori tra -1 e +1)
3. Salva: `FI_UVR` si aggiorna automaticamente via formula

---

## CELLA 1 — Montaggio Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Google Drive montato')

Mounted at /content/drive
✅ Google Drive montato


## CELLA 2 — Installazione dipendenze

In [ ]:
!pip install -q openpyxl pandas
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
import os
print('✅ Dipendenze caricate')

✅ Dipendenze caricate


## CELLA 3 — ✏️ Configurazione

**Modifica solo questa cella** con i tuoi percorsi.

In [ ]:
# =============================================================
# ✏️  MODIFICA QUI
# =============================================================

# Percorso del file Excel di input (Object Detection + Mapillary)
INPUT_EXCEL_PATH = '...'

# Cartella dove salvare l'Excel di output
OUTPUT_FOLDER = '...'

# Nome del file di output
OUTPUT_EXCEL_NAME = '...'

# Nome del foglio nel file di input
INPUT_SHEET_NAME = '...'

# Valore nella colonna 'Orientamento' che identifica le righe riassuntive
SUMMARY_ROW_LABEL = 'TOTALE'

# =============================================================
# Colonne fisse (NON classi di oggetti) — vengono escluse automaticamente
# dal rilevamento delle classi. Modifica solo se il tuo file ha nomi diversi.
# =============================================================
FIXED_COLUMNS = ['Latitudine', 'Longitudine', 'PANO_ID', 'Orientamento']

# =============================================================

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print('✅ Configurazione completata')
print(f'   Input:  {INPUT_EXCEL_PATH}')
print(f'   Output: {OUTPUT_FOLDER}/{OUTPUT_EXCEL_NAME}')

✅ Configurazione completata
   Input:  /content/drive/MyDrive/Polimi/SVBEF/Progetto Finale/c_Definitivo/c_Outputs_Excel Script Mapillary - Inputs_Excel Script Excel con FI/Results_ObjectDetection+Mapillary Definitivo.xlsx
   Output: /content/drive/MyDrive/Polimi/SVBEF/Progetto Finale/c_Definitivo/d_Creazione Excel con FI/flexibility_index.xlsx


## CELLA 4 — Lettura del file Excel e rilevamento automatico delle classi

In [ ]:
# Leggi tutto il file
df_all = pd.read_excel(INPUT_EXCEL_PATH, sheet_name=INPUT_SHEET_NAME, dtype=str)

print(f'✅ File Excel letto')
print(f'   Righe totali: {len(df_all)}')
print(f'   Colonne ({len(df_all.columns)}): {list(df_all.columns)}')

# Verifica colonne fisse
for col in FIXED_COLUMNS:
    if col not in df_all.columns:
        print(f'⚠️  Colonna fissa "{col}" non trovata — verrà ignorata')

# Rileva automaticamente le colonne-classe
# = tutto ciò che non è in FIXED_COLUMNS
CLASS_COLUMNS = [c for c in df_all.columns if c not in FIXED_COLUMNS]

# Isola le righe TOTALE
mask_totale = df_all['Orientamento'].str.strip() == SUMMARY_ROW_LABEL
df_summary  = df_all[mask_totale].copy().reset_index(drop=True)

# Converti le colonne-classe in numerico (NaN → 0)
for col in CLASS_COLUMNS:
    df_summary[col] = pd.to_numeric(df_summary[col], errors='coerce').fillna(0).astype(int)

# Converti lat/lng
df_summary['Latitudine']  = pd.to_numeric(df_summary['Latitudine'],  errors='coerce')
df_summary['Longitudine'] = pd.to_numeric(df_summary['Longitudine'], errors='coerce')

print(f'\n✅ Righe TOTALE (punti geografici): {len(df_summary)}')
print(f'   Classi rilevate ({len(CLASS_COLUMNS)}): {CLASS_COLUMNS}')
print(f'\n📋 Anteprima (prime 5 righe TOTALE):')
print(df_summary[['PANO_ID', 'Latitudine', 'Longitudine'] + CLASS_COLUMNS].head().to_string(index=False))

✅ File Excel letto
   Righe totali: 23550
   Colonne (20): ['Latitudine', 'Longitudine', 'PANO_ID', 'Orientamento', 'awning', 'canopy', 'fence', 'billboard', 'bus stop shelter', 'sculpture', 'monument', 'fountain', 'umbrella', 'mly_Banner', 'mly_Bench', 'mly_Phone_Booth', 'mly_Sign_Store', 'mly_Pole', 'mly_Utility_Pole', 'mly_Trash_Can']

✅ Righe TOTALE (punti geografici): 4710
   Classi rilevate (16): ['awning', 'canopy', 'fence', 'billboard', 'bus stop shelter', 'sculpture', 'monument', 'fountain', 'umbrella', 'mly_Banner', 'mly_Bench', 'mly_Phone_Booth', 'mly_Sign_Store', 'mly_Pole', 'mly_Utility_Pole', 'mly_Trash_Can']

📋 Anteprima (prime 5 righe TOTALE):
               PANO_ID  Latitudine  Longitudine  awning  canopy  fence  billboard  bus stop shelter  sculpture  monument  fountain  umbrella  mly_Banner  mly_Bench  mly_Phone_Booth  mly_Sign_Store  mly_Pole  mly_Utility_Pole  mly_Trash_Can
--tlUSu9Lrgn80E_kwY5-g   45.451178     9.206384       0       1      0          2           

## CELLA 5 — Costruzione del file Excel con pesi editabili e indice FI_VBR

Genera il file Excel strutturato in un unico foglio:
- **Sezione A** (righe 1–N): tabella pesi — inserisci qui i pesi UVR per ogni classe
- **Sezione B** (righe successive): conteggi per punto geografico + FI_UVR via formula Excel

Le formule Excel si aggiornano automaticamente quando modifichi i pesi nella Sezione A.

In [ ]:
output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_EXCEL_NAME)
wb = Workbook()
ws = wb.active
ws.title = 'UVR Index'

# ================================================================
# STILI
# ================================================================
def make_fill(hex_color):
    return PatternFill(start_color=hex_color, end_color=hex_color, fill_type='solid')

def make_border(style='thin', color='BFBFBF'):
    s = Side(style=style, color=color)
    return Border(left=s, right=s, top=s, bottom=s)

C_DARK_BLUE   = '1F4E79'
C_MID_BLUE    = '2E75B6'
C_LIGHT_BLUE  = 'DEEAF1'
C_MID_GREEN   = '70AD47'
C_LIGHT_GREEN = 'E2EFDA'
C_GOLD        = 'FFC000'
C_LIGHT_GOLD  = 'FFF2CC'
C_GRAY_LIGHT  = 'F2F2F2'
C_WHITE       = 'FFFFFF'

def style_header(cell, bg, fg='FFFFFF', bold=True, size=10, wrap=False, h_align='center'):
    cell.fill      = make_fill(bg)
    cell.font      = Font(bold=bold, color=fg, name='Arial', size=size)
    cell.alignment = Alignment(horizontal=h_align, vertical='center', wrap_text=wrap)
    cell.border    = make_border('thin', 'FFFFFF')

def style_cell(cell, bg=C_WHITE, fg='000000', bold=False, size=10,
               h_align='center', number_format=None, border_color='BFBFBF'):
    cell.fill      = make_fill(bg)
    cell.font      = Font(bold=bold, color=fg, name='Arial', size=size)
    cell.alignment = Alignment(horizontal=h_align, vertical='center')
    cell.border    = make_border('thin', border_color)
    if number_format:
        cell.number_format = number_format

n_classes = len(CLASS_COLUMNS)

# ================================================================
# SEZIONE A — TABELLA PESI
# Riga 1 = titolo, Riga 2 = intestazioni, Righe 3..N+2 = classi
# Colonna A = nome classe, Colonna B = peso UVR (editabile)
# ================================================================
ROW_SECTION_A_TITLE = 1
ROW_WEIGHTS_HEADER  = 2
ROW_WEIGHTS_FIRST   = 3
ROW_WEIGHTS_LAST    = ROW_WEIGHTS_FIRST + n_classes - 1

# Titolo sezione A
ws.merge_cells(start_row=ROW_SECTION_A_TITLE, start_column=1,
               end_row=ROW_SECTION_A_TITLE, end_column=2)
cell = ws.cell(row=ROW_SECTION_A_TITLE, column=1,
               value='SEZIONE A — PESI PER CLASSE  (inserisci valori tra -1 e +1)')
style_header(cell, C_DARK_BLUE, size=11)
ws.row_dimensions[ROW_SECTION_A_TITLE].height = 22

# Intestazioni
for c_idx, (hdr, bg) in enumerate(
        [('Classe (oggetto)', C_MID_BLUE), ('Peso VBR', C_MID_GREEN)], start=1):
    cell = ws.cell(row=ROW_WEIGHTS_HEADER, column=c_idx, value=hdr)
    style_header(cell, bg)
ws.row_dimensions[ROW_WEIGHTS_HEADER].height = 20

# Righe pesi
for i, cls in enumerate(CLASS_COLUMNS):
    r      = ROW_WEIGHTS_FIRST + i
    bg_row = C_LIGHT_BLUE if i % 2 == 0 else C_WHITE
    c = ws.cell(row=r, column=1, value=cls)
    style_cell(c, bg=bg_row, bold=True, h_align='left')
    c = ws.cell(row=r, column=2, value=0.0)
    style_cell(c, bg=C_LIGHT_GREEN, number_format='0.00', border_color=C_MID_GREEN)
    ws.row_dimensions[r].height = 18

# Nota esplicativa
ROW_NOTE = ROW_WEIGHTS_LAST + 1
ws.merge_cells(start_row=ROW_NOTE, start_column=1, end_row=ROW_NOTE, end_column=2)
nc = ws.cell(row=ROW_NOTE, column=1,
             value='ℹ️  Pesi da -1 a +1. Negativi = impatto negativo. '
                   'Salva dopo aver inserito i pesi: FI_VBR si aggiorna automaticamente.')
nc.fill      = make_fill('FFFBDE')
nc.font      = Font(italic=True, color='7F6000', name='Arial', size=9)
nc.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
ws.row_dimensions[ROW_NOTE].height = 28

# Separatore
ROW_SEPARATOR = ROW_NOTE + 1
ws.row_dimensions[ROW_SEPARATOR].height = 8

# ================================================================
# SEZIONE B — DATI PER PUNTO GEOGRAFICO + FI_UVR
# ================================================================
ROW_SECTION_B_TITLE = ROW_SEPARATOR + 1
ROW_DATA_HEADER     = ROW_SECTION_B_TITLE + 1
ROW_DATA_FIRST      = ROW_DATA_HEADER + 1

# Layout colonne sezione B
COL_PANO        = 1
COL_LAT         = 2
COL_LNG         = 3
COL_CLASS_FIRST = 4
COL_CLASS_LAST  = COL_CLASS_FIRST + n_classes - 1
COL_FI_VBR     = COL_CLASS_LAST + 1
COL_VBR_PCT    = COL_CLASS_LAST + 2

# Titolo sezione B
ws.merge_cells(start_row=ROW_SECTION_B_TITLE, start_column=1,
               end_row=ROW_SECTION_B_TITLE, end_column=COL_VBR_PCT)
cell = ws.cell(row=ROW_SECTION_B_TITLE, column=1,
               value='SEZIONE B — CONTEGGI PER PUNTO GEOGRAFICO E FLEXIBILITY INDEX')
style_header(cell, C_DARK_BLUE, size=11)
ws.row_dimensions[ROW_SECTION_B_TITLE].height = 22

# Intestazioni sezione B
for c_idx, (hdr, bg) in enumerate(
        [('PANO_ID', C_MID_BLUE), ('Latitudine', C_MID_BLUE), ('Longitudine', C_MID_BLUE)],
        start=1):
    cell = ws.cell(row=ROW_DATA_HEADER, column=c_idx, value=hdr)
    style_header(cell, bg)

for i, cls in enumerate(CLASS_COLUMNS):
    cell = ws.cell(row=ROW_DATA_HEADER, column=COL_CLASS_FIRST + i, value=cls)
    style_header(cell, C_MID_BLUE, wrap=True)

cell = ws.cell(row=ROW_DATA_HEADER, column=COL_FI_VBR, value='FI_VBR')
style_header(cell, C_MID_GREEN, size=11)

cell = ws.cell(row=ROW_DATA_HEADER, column=COL_VBR_PCT, value='VBR%')
style_header(cell, C_GOLD, fg='000000', size=11)

ws.row_dimensions[ROW_DATA_HEADER].height = 30

# Calcola il range FI_UVR per la formula UVR%
n_data_rows        = len(df_summary)
fi_col_letter      = get_column_letter(COL_FI_VBR)
fi_range_first     = ROW_DATA_FIRST
fi_range_last      = ROW_DATA_FIRST + n_data_rows - 1
fi_range_abs       = f'${fi_col_letter}${fi_range_first}:${fi_col_letter}${fi_range_last}'

# ================================================================
# RIGHE DATI
# ================================================================
for row_idx, (_, row_data) in enumerate(df_summary.iterrows()):
    r      = ROW_DATA_FIRST + row_idx
    bg_row = C_GRAY_LIGHT if row_idx % 2 == 0 else C_WHITE

    # PANO_ID
    c = ws.cell(row=r, column=COL_PANO, value=str(row_data['PANO_ID']))
    style_cell(c, bg=bg_row, h_align='left')

    # Latitudine
    try:
        lat_val = float(row_data['Latitudine'])
    except (ValueError, TypeError):
        lat_val = row_data['Latitudine']
    c = ws.cell(row=r, column=COL_LAT, value=lat_val)
    style_cell(c, bg=bg_row, number_format='0.0000000')

    # Longitudine
    try:
        lng_val = float(row_data['Longitudine'])
    except (ValueError, TypeError):
        lng_val = row_data['Longitudine']
    c = ws.cell(row=r, column=COL_LNG, value=lng_val)
    style_cell(c, bg=bg_row, number_format='0.0000000')

    # Conteggi per classe
    for i, cls in enumerate(CLASS_COLUMNS):
        count = int(row_data.get(cls, 0))
        c = ws.cell(row=r, column=COL_CLASS_FIRST + i, value=count)
        style_cell(c, bg=bg_row, number_format='0')

    # FI_UVR — formula: sommatoria(peso_i * conteggio_i)
    # Peso della classe i è in $B$(ROW_WEIGHTS_FIRST + i)
    # Conteggio è nella cella della stessa riga, colonna COL_CLASS_FIRST + i
    terms = [
        f'$B${ROW_WEIGHTS_FIRST + i}*{get_column_letter(COL_CLASS_FIRST + i)}{r}'
        for i in range(n_classes)
    ]
    c = ws.cell(row=r, column=COL_FI_VBR, value='=' + '+'.join(terms))
    style_cell(c, bg=C_LIGHT_GREEN, bold=True, number_format='0.000',
               border_color=C_MID_GREEN)

    # UVR% — FI_UVR / MAX(tutti FI_UVR) * 100  (0 se MAX=0)
    fi_cell = f'{fi_col_letter}{r}'
    formula_pct = f'=IF(MAX({fi_range_abs})=0,0,{fi_cell}/MAX({fi_range_abs})*100)'
    c = ws.cell(row=r, column=COL_VBR_PCT, value=formula_pct)
    style_cell(c, bg=C_LIGHT_GOLD, bold=True, number_format='0.00',
               border_color=C_GOLD)

    ws.row_dimensions[r].height = 18

# ================================================================
# LARGHEZZE COLONNE
# ================================================================
ws.column_dimensions['A'].width = 32   # PANO_ID / Classe
ws.column_dimensions['B'].width = 12   # Peso VBR / Latitudine
ws.column_dimensions[get_column_letter(COL_LNG)].width = 12
for i in range(n_classes):
    ws.column_dimensions[get_column_letter(COL_CLASS_FIRST + i)].width = \
        max(10, len(CLASS_COLUMNS[i]) + 2)
ws.column_dimensions[get_column_letter(COL_FI_VBR)].width  = 14
ws.column_dimensions[get_column_letter(COL_VBR_PCT)].width = 12

# ================================================================
# FREEZE PANES — blocca intestazioni sezione B
# ================================================================
ws.freeze_panes = ws.cell(row=ROW_DATA_FIRST, column=COL_CLASS_FIRST)

# ================================================================
# VALIDAZIONE DATI — pesi tra -1 e 1
# ================================================================
dv = DataValidation(
    type='decimal', operator='between', formula1='-1', formula2='1',
    showErrorMessage=True, errorTitle='Valore non valido',
    error='Il peso deve essere compreso tra -1 e +1.',
    showInputMessage=True, promptTitle='Peso VBR',
    prompt='Inserisci un valore decimale tra -1 e +1'
)
dv.sqref = f'B{ROW_WEIGHTS_FIRST}:B{ROW_WEIGHTS_LAST}'
ws.add_data_validation(dv)

# ================================================================
# SALVA
# ================================================================
wb.save(output_path)

print(f'✅ File Excel salvato: {output_path}')
print(f'\n📋 Struttura:')
print(f'   Righe  1–{ROW_WEIGHTS_LAST:<4} → Sezione A: pesi VBR (editabili)')
print(f'   Riga  {ROW_NOTE:<5} → Nota')
print(f'   Righe {ROW_SECTION_B_TITLE}–{ROW_DATA_FIRST-1:<4} → Sezione B: intestazioni')
print(f'   Righe {ROW_DATA_FIRST}–{ROW_DATA_FIRST + n_data_rows - 1:<4} → Dati: {n_data_rows} punti geografici')
print(f'\n   Classi ({n_classes}): {CLASS_COLUMNS}')
print(f'   Punti geografici: {n_data_rows}')
print(f'\n➡️  Apri l\'Excel, inserisci i pesi nella Sezione A e salva.')
print(f'   FI_VBR e VBR% si aggiornano automaticamente.')

✅ File Excel salvato: /content/drive/MyDrive/Polimi/SVBEF/Progetto Finale/c_Definitivo/d_Creazione Excel con FI/flexibility_index.xlsx

📋 Struttura:
   Righe  1–18   → Sezione A: pesi VBR (editabili)
   Riga  19    → Nota
   Righe 21–22   → Sezione B: intestazioni
   Righe 23–4732 → Dati: 4710 punti geografici

   Classi (16): ['awning', 'canopy', 'fence', 'billboard', 'bus stop shelter', 'sculpture', 'monument', 'fountain', 'umbrella', 'mly_Banner', 'mly_Bench', 'mly_Phone_Booth', 'mly_Sign_Store', 'mly_Pole', 'mly_Utility_Pole', 'mly_Trash_Can']
   Punti geografici: 4710

➡️  Apri l'Excel, inserisci i pesi nella Sezione A e salva.
   FI_VBR e VBR% si aggiornano automaticamente.


## CELLA 6 — (Facoltativa) Ricalcolo FI_UVR in Python dopo aver inserito i pesi

Dopo aver inserito i pesi nell'Excel e salvato, riesegui questa cella per:
- Leggere i pesi aggiornati dal file
- Ricalcolare FI_UVR in Python
- Stampare il riepilogo finale a schermo

In [ ]:
# Rileggi i pesi dal file Excel (dopo averli inseriti manualmente)
wb_read = load_workbook(output_path, data_only=True)
ws_read = wb_read['UVR Index']

weights_vbr = {}
for i, cls in enumerate(CLASS_COLUMNS):
    r   = ROW_WEIGHTS_FIRST + i
    val = ws_read.cell(row=r, column=2).value
    weights_vbr[cls] = float(val) if val is not None else 0.0

print('📊 Pesi letti dal file Excel:')
print(f"  {'CLASSE':<30} {'PESO VBR':>10}")
print('  ' + '-' * 42)
for cls, w in weights_vbr.items():
    print(f"  {cls:<30} {w:>10.3f}")

# Ricalcola FI_UVR
df_summary['FI_UVR'] = sum(
    df_summary[cls].astype(float) * weights_vbr[cls]
    for cls in CLASS_COLUMNS
)
max_fi = df_summary['FI_UVR'].max()
df_summary['UVR%'] = df_summary['FI_UVR'].apply(
    lambda x: (x / max_fi * 100) if max_fi != 0 else 0
)

df_final = df_summary[['PANO_ID', 'Latitudine', 'Longitudine', 'FI_VBR', 'UVR%']].copy()

print(f'\n🎯 UVR INDEX — {len(df_final)} punti geografici:')
print(df_final.to_string(index=False))

print(f'\n📈 Statistiche FI_UVR:')
print(f'   Min:   {df_final["FI_UVR"].min():.3f}')
print(f'   Max:   {df_final["FI_UVR"].max():.3f}')
print(f'   Media: {df_final["FI_UVR"].mean():.3f}')

📊 Pesi letti dal file Excel:
  CLASSE                           PESO VBR
  ------------------------------------------
  awning                              0.500
  canopy                              0.000
  fence                               0.000
  billboard                           0.000
  bus stop shelter                    0.000
  sculpture                           0.000
  monument                            0.000
  fountain                            0.000
  umbrella                            0.000
  mly_Banner                          0.000
  mly_Bench                           0.000
  mly_Phone_Booth                     0.000
  mly_Sign_Store                      0.000
  mly_Pole                            0.000
  mly_Utility_Pole                    0.000
  mly_Trash_Can                       0.000

🎯 FLEXIBILITY INDEX — 4710 punti geografici:
                             PANO_ID  Latitudine  Longitudine  FI_VBR  VBR%
              --tlUSu9Lrgn80E_kwY5-g   45.451178     9.2